![Logo](https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/shared_assets/logo.png)

**Developers:** Zoltan Barta  
**Date:** 2026-04-05  
**Version:** 2025-26/2

[<img src="https://colab.research.google.com/assets/colab-badge.svg">](https://colab.research.google.com/github/BartaZoltan/deep-reinforcement-learning-course/blob/main/notebooks/sessions/8_dqn_improvements_rainbow/dqn_improvements_rainbow.ipynb)

# Practice 8: Rainbow DQN Components


## Summary

In Practice 7, we built a working `DQN` pipeline on Pong and then improved only the bootstrap target to obtain `Double DQN`. This session continues that line in the same environment, but now the focus shifts from one target correction to the rest of the major Rainbow ingredients.

This notebook follows one coherent progression:
- prioritized replay,
- dueling networks,
- multi-step returns inside deep Q-learning,
- noisy networks for learned exploration,
- distributional value prediction,
- and a final block that shows how these pieces fit together into a Rainbow-style update.

The goal is not to hide Rainbow behind one giant opaque implementation. Instead, each ingredient is introduced separately with a short showcase, a code block, and a small sanity-check cell.


## From Double DQN to Rainbow

Rainbow is best understood as a **combination** of several ideas rather than a completely separate value-based method. The core template from DQN is still there: replay, target networks, bootstrapped value updates, and a convolutional network that maps the Pong frame stack to action values. What changes is that each part of this pipeline becomes a little stronger.

That is why the notebook is organized by improvements rather than by one monolithic final algorithm. If each ingredient is understandable on its own, then the combined Rainbow training step at the end becomes much easier to read.


## Session Setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import urllib.request

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
except Exception:
    torch = None
    nn = None
    F = None
    optim = None


def _load_session8_utils():
    candidates = [
        Path('utils.py'),
        Path('notebooks/sessions/8_dqn_improvements_rainbow/utils.py'),
        Path('/content/notebooks/sessions/8_dqn_improvements_rainbow/utils.py'),
    ]

    utils_path = next((p for p in candidates if p.exists()), None)
    if utils_path is None:
        utils_path = Path('notebooks/sessions/8_dqn_improvements_rainbow/utils.py')
        utils_path.parent.mkdir(parents=True, exist_ok=True)
        url = (
            'https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/'
            'notebooks/sessions/8_dqn_improvements_rainbow/utils.py'
        )
        urllib.request.urlretrieve(url, utils_path)

    spec = importlib.util.spec_from_file_location('session8_utils', utils_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


if torch is None:
    raise RuntimeError('PyTorch is required for this notebook.')

s8u = _load_session8_utils()
SEED = 42
GLOBAL_RNG = s8u.set_seed(SEED)
DEVICE = s8u.select_device()
ASSET_ROOT = s8u.resolve_asset_root('notebooks/sessions/8_dqn_improvements_rainbow')

PONG_OBS_SHAPE = (4, 84, 84)
PONG_NUM_ACTIONS = 6
PONG_ACTION_NAMES = ['NOOP', 'FIRE', 'RIGHT', 'LEFT', 'RIGHTFIRE', 'LEFTFIRE']

print('Device:', DEVICE)
print('Reference observation shape:', PONG_OBS_SHAPE)
print('Reference action space size:', PONG_NUM_ACTIONS)


### Continuing with Pong

This practice stays in the same Atari setting as Practice 7. The reference observation is still a stacked Pong state with shape `(4, 84, 84)`, and the action space still has `6` discrete actions. The difference is that here we will mostly study the **improvements around the DQN pipeline**, not the environment setup itself.

That is why the cells below use lightweight toy examples and shape checks instead of full Atari training runs. The aim is to understand what each Rainbow ingredient changes before combining them again at the end.


In [ ]:
s8u.print_rainbow_component_table()
print('Pong action names:', PONG_ACTION_NAMES)


## Prioritized Experience Replay


Uniform replay treats every transition in the buffer equally. Prioritized replay changes this by sampling transitions with large TD errors more often, because those are usually the ones the current network still understands poorly.

This introduces a second ingredient as well: **importance-sampling weights**. If the sampling distribution is no longer uniform, the loss should be reweighted so that the update stays better behaved. In practice, prioritized replay is therefore always a pair: non-uniform sampling plus a corrective weight in the loss.

Reference paper: [Schaul et al. (2015), *Prioritized Experience Replay*](https://arxiv.org/abs/1511.05952).


In [ ]:
toy_td_errors = np.array([0.02, 0.10, 0.25, 0.55, 0.90, 0.15, 0.05, 0.40])
priority_demo = s8u.plot_prioritized_replay_demo(toy_td_errors, alpha=0.6, beta=0.4)
print('Sampling probabilities:', np.round(priority_demo['probabilities'], 3))
print('Importance weights:', np.round(priority_demo['weights'], 3))


### Task 1

**Implement a proportional prioritized replay buffer (20 min)**

The goal is to keep the same transition storage as in DQN, but add two new pieces of logic.

- Each stored transition must carry a priority.
- Sampling should use those priorities instead of a uniform distribution.

The code below uses the simpler notebook-friendly version based on direct probability computation rather than a sum-tree. That is not the fastest implementation, but it keeps the main idea visible:

- priorities are exponentiated by `alpha`,
- normalized into sampling probabilities,
- sampled with `np.random.choice`,
- and corrected with importance-sampling weights controlled by `beta`.

Only the sampling logic matters here. The buffer should still return the same transition tensors as before, together with the sampled indices and the normalized importance weights needed later in the loss.


In [ ]:
class PrioritizedReplayBuffer:
    def __init__(self, capacity: int, obs_shape: tuple[int, ...], device, alpha: float = 0.6, eps: float = 1e-6):
        self.capacity = int(capacity)
        self.device = device
        self.alpha = float(alpha)
        self.eps = float(eps)
        self.obs = np.zeros((capacity, *obs_shape), dtype=np.uint8)
        self.next_obs = np.zeros((capacity, *obs_shape), dtype=np.uint8)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.dones = np.zeros(capacity, dtype=np.float32)
        self.priorities = np.zeros(capacity, dtype=np.float32)
        self.pos = 0
        self.size = 0

    def add(self, obs, action: int, reward: float, next_obs, done: bool, priority: float | None = None):
########################################################################
        self.obs[self.pos] = np.asarray(obs, dtype=np.uint8)
        self.next_obs[self.pos] = np.asarray(next_obs, dtype=np.uint8)
        self.actions[self.pos] = int(action)
        self.rewards[self.pos] = float(reward)
        self.dones[self.pos] = float(done)

        if priority is None:
            max_priority = float(self.priorities[: self.size].max()) if self.size > 0 else 1.0
            priority = max_priority
        self.priorities[self.pos] = float(priority)

        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)
########################################################################

    def sample(self, batch_size: int, beta: float = 0.4) -> dict[str, torch.Tensor]:
########################################################################
        if self.size == 0:
            raise ValueError('Cannot sample from an empty buffer.')

        valid_priorities = self.priorities[: self.size] + self.eps
        probs = valid_priorities ** self.alpha
        probs = probs / probs.sum()
        indices = np.random.choice(self.size, size=batch_size, replace=self.size < batch_size, p=probs)

        weights = (self.size * probs[indices]) ** (-beta)
        weights = weights / weights.max()

        return {
            'obs': torch.as_tensor(self.obs[indices], dtype=torch.float32, device=self.device) / 255.0,
            'actions': torch.as_tensor(self.actions[indices], dtype=torch.long, device=self.device),
            'rewards': torch.as_tensor(self.rewards[indices], dtype=torch.float32, device=self.device),
            'next_obs': torch.as_tensor(self.next_obs[indices], dtype=torch.float32, device=self.device) / 255.0,
            'dones': torch.as_tensor(self.dones[indices], dtype=torch.float32, device=self.device),
            'weights': torch.as_tensor(weights, dtype=torch.float32, device=self.device),
            'indices': torch.as_tensor(indices, dtype=torch.long, device=self.device),
            'probabilities': torch.as_tensor(probs[indices], dtype=torch.float32, device=self.device),
        }
########################################################################

    def update_priorities(self, indices, priorities):
########################################################################
        indices = np.asarray(indices, dtype=np.int64)
        priorities = np.asarray(priorities, dtype=np.float32)
        self.priorities[indices] = priorities
########################################################################

    def __len__(self) -> int:
        return self.size


In [ ]:
per_buffer = PrioritizedReplayBuffer(capacity=16, obs_shape=PONG_OBS_SHAPE, device=DEVICE, alpha=0.6)

for idx in range(8):
    obs = np.full(PONG_OBS_SHAPE, idx, dtype=np.uint8)
    next_obs = np.full(PONG_OBS_SHAPE, idx + 1, dtype=np.uint8)
    per_buffer.add(obs, idx % PONG_NUM_ACTIONS, reward=float(idx % 3 - 1), next_obs=next_obs, done=bool(idx == 7), priority=0.1 * (idx + 1))

per_batch = per_buffer.sample(batch_size=4, beta=0.4)
print('Sampled obs shape:', tuple(per_batch['obs'].shape))
print('Sampled indices:', per_batch['indices'].tolist())
print('Sampled weights:', torch.round(per_batch['weights'], decimals=3))
print('Sampled probabilities:', torch.round(per_batch['probabilities'], decimals=3))

per_buffer.update_priorities(per_batch['indices'].cpu().numpy(), np.array([1.2, 0.9, 0.7, 1.5], dtype=np.float32))
print('Updated priorities at sampled indices.')


## Dueling Networks


A standard Q-network predicts one score per action directly. A dueling network first separates that prediction into two streams:

- a **state-value stream** `V(s)`,
- and an **advantage stream** `A(s, a)`.

These are then recombined into action values. The main benefit is that the network can learn that a state is generally good or bad even before it knows large action-specific differences. In Atari this often improves efficiency, because many frames clearly look good or bad overall, while the relative action differences are smaller.

Reference paper: [Wang et al. (2016), *Dueling Network Architectures for Deep Reinforcement Learning*](https://arxiv.org/abs/1511.06581).


In [ ]:
dueling_demo = s8u.plot_dueling_decomposition(
    value=1.2,
    advantages=[-0.4, 0.0, 0.6, -0.1, 0.2, -0.3],
    action_labels=PONG_ACTION_NAMES,
)
print('Centered advantages:', np.round(dueling_demo['centered_advantages'], 3))
print('Resulting Q-values:', np.round(dueling_demo['q_values'], 3))


### Task 2

**Implement a dueling convolutional Q-network (20 min)**

The convolutional backbone can stay almost identical to the DQN network from Practice 7. The change happens only after the shared visual features have been extracted.

- One head should predict a single scalar `V(s)` per state.
- One head should predict `A(s, a)` for all actions.
- The final Q-values should be combined as
  $$
  Q(s, a) = V(s) + A(s, a) - \frac{1}{|\mathcal{A}|} \sum_{a'} A(s, a').
  $$

Subtracting the mean keeps the decomposition identifiable. That is the key line to get right.


In [ ]:
class DuelingQNetwork(nn.Module):
    def __init__(self, obs_shape: tuple[int, int, int], num_actions: int):
        super().__init__()
        c, h, w = obs_shape
        self.features = nn.Sequential(
            nn.Conv2d(c, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, c, h, w)
            feature_dim = int(np.prod(self.features(dummy).shape[1:]))

        self.value_stream = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Linear(512, num_actions),
        )

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
########################################################################
        features = self.features(obs).flatten(start_dim=1)
        values = self.value_stream(features)
        advantages = self.advantage_stream(features)
        return values + advantages - advantages.mean(dim=1, keepdim=True)
########################################################################


In [ ]:
dueling_net = DuelingQNetwork(PONG_OBS_SHAPE, PONG_NUM_ACTIONS).to(DEVICE)
dummy_obs = torch.rand(2, *PONG_OBS_SHAPE, device=DEVICE)
dummy_q = dueling_net(dummy_obs)
print('Dueling network output shape:', tuple(dummy_q.shape))


## Multi-step Returns inside Rainbow


Rainbow also keeps **multi-step targets**. This is one of the places where the deep value-based line reconnects to Practice 5. The idea is exactly the same as before: do not bootstrap after only one reward if a slightly longer backup can propagate information faster.

In deep Q-learning, this means that replay items are often constructed from short reward sequences rather than only one-step transitions. The target becomes
$$
R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^n \max_{a'} Q(S_{t+n}, a').
$$
That longer backup is especially helpful in sparse-reward Atari games where one-step signals can move too slowly.

Reference papers: [Mnih et al. (2016), *Asynchronous Methods for Deep Reinforcement Learning*](https://arxiv.org/abs/1602.01783) for deep n-step targets, and [Hessel et al. (2018), *Rainbow: Combining Improvements in Deep Reinforcement Learning*](https://arxiv.org/abs/1710.02298) for their use inside Rainbow.


In [ ]:
def compute_n_step_targets(
    reward_sequences: torch.Tensor,
    bootstrap_values: torch.Tensor,
    dones: torch.Tensor,
    gamma: float,
) -> torch.Tensor:
    targets = torch.zeros(reward_sequences.shape[0], dtype=reward_sequences.dtype, device=reward_sequences.device)
    discount = 1.0
    for step_idx in range(reward_sequences.shape[1]):
        targets += discount * reward_sequences[:, step_idx]
        discount *= gamma
    targets += discount * (1.0 - dones) * bootstrap_values
    return targets


reward_sequences = torch.tensor([[-1.0, 0.0, 1.0], [-1.0, -1.0, 0.0]], device=DEVICE)
bootstrap_values = torch.tensor([0.8, 0.3], device=DEVICE)
dones = torch.tensor([0.0, 1.0], device=DEVICE)
n_step_targets = compute_n_step_targets(reward_sequences, bootstrap_values, dones, gamma=0.99)

n_step_demo = s8u.plot_n_step_targets(rewards=[-1.0, 0.0, 1.0, 0.5, -0.5], gamma=0.99, bootstrap_value=0.8, n_values=(1, 3, 5))
print('Example 3-step targets:', torch.round(n_step_targets, decimals=3))
print('Toy comparison targets:', np.round(n_step_demo['targets'], 3))


## Noisy Networks


DQN usually relies on an external exploration schedule such as epsilon-greedy. Noisy networks replace that with **learned parameter noise** inside the network itself. Instead of manually injecting random actions from outside, the Q-network becomes stochastic in a structured way: different forward passes can produce slightly different action preferences, and the scale of that noise is learned.

This is attractive in Atari because the exploration mechanism stays tied to the value function rather than being added as a separate policy rule.

Reference paper: [Fortunato et al. (2017), *Noisy Networks for Exploration*](https://arxiv.org/abs/1706.10295).


### Task 3

**Implement a factorized NoisyLinear layer (20 min)**

The main idea is that the weight matrix and bias become random variables during training:

- a learned mean parameter,
- plus a learned standard deviation parameter,
- multiplied by sampled noise.

At evaluation time, the layer should fall back to the deterministic mean parameters. The code below uses the factorized Gaussian version from the NoisyNet paper, where one noise vector is sampled for the input dimension and one for the output dimension, then combined by an outer product.


In [ ]:
class NoisyLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, sigma_init: float = 0.5):
        super().__init__()
        self.in_features = int(in_features)
        self.out_features = int(out_features)

        self.weight_mu = nn.Parameter(torch.empty(out_features, in_features))
        self.weight_sigma = nn.Parameter(torch.empty(out_features, in_features))
        self.bias_mu = nn.Parameter(torch.empty(out_features))
        self.bias_sigma = nn.Parameter(torch.empty(out_features))

        self.register_buffer('weight_epsilon', torch.zeros(out_features, in_features))
        self.register_buffer('bias_epsilon', torch.zeros(out_features))
        self.sigma_init = float(sigma_init)
        self.reset_parameters()
        self.reset_noise()

    def reset_parameters(self):
        mu_range = 1.0 / np.sqrt(self.in_features)
        self.weight_mu.data.uniform_(-mu_range, mu_range)
        self.bias_mu.data.uniform_(-mu_range, mu_range)
        self.weight_sigma.data.fill_(self.sigma_init / np.sqrt(self.in_features))
        self.bias_sigma.data.fill_(self.sigma_init / np.sqrt(self.out_features))

    @staticmethod
    def _scale_noise(size: int) -> torch.Tensor:
        x = torch.randn(size)
        return x.sign() * x.abs().sqrt()

    def reset_noise(self):
########################################################################
        eps_in = self._scale_noise(self.in_features).to(self.weight_mu.device)
        eps_out = self._scale_noise(self.out_features).to(self.weight_mu.device)
        self.weight_epsilon.copy_(eps_out.outer(eps_in))
        self.bias_epsilon.copy_(eps_out)
########################################################################

    def forward(self, x: torch.Tensor) -> torch.Tensor:
########################################################################
        if self.training:
            weight = self.weight_mu + self.weight_sigma * self.weight_epsilon
            bias = self.bias_mu + self.bias_sigma * self.bias_epsilon
        else:
            weight = self.weight_mu
            bias = self.bias_mu
        return F.linear(x, weight, bias)
########################################################################


In [ ]:
noisy_layer = NoisyLinear(128, PONG_NUM_ACTIONS).to(DEVICE)
input_vector = torch.randn(1, 128, device=DEVICE)

samples = []
noisy_layer.train()
for _ in range(6):
    noisy_layer.reset_noise()
    samples.append(noisy_layer(input_vector).detach().cpu().numpy()[0])

samples = np.stack(samples)
s8u.plot_noisy_action_value_samples(samples, action_labels=PONG_ACTION_NAMES)

noisy_layer.eval()
print('Deterministic evaluation output:', torch.round(noisy_layer(input_vector).detach().cpu(), decimals=3))


## Distributional DQN (C51)


Standard DQN predicts only the **expected return** for each action. Distributional DQN predicts a full categorical distribution over a fixed support of return atoms. The action value is still available as the expectation of that distribution, but the network now has access to richer information about uncertainty and outcome shape.

In the C51 formulation, each action predicts probabilities over a fixed support
$$
z_1, z_2, \ldots, z_N.
$$
The target is then projected back onto this fixed support after the Bellman update. This projection step is the distinctive part of the method.

Reference paper: [Bellemare, Dabney, and Munos (2017), *A Distributional Perspective on Reinforcement Learning*](https://arxiv.org/abs/1707.06887).


In [ ]:
toy_support = np.linspace(-10, 10, 11)
toy_probs = np.array([0.01, 0.03, 0.06, 0.10, 0.16, 0.22, 0.18, 0.12, 0.07, 0.03, 0.02])
toy_probs = toy_probs / toy_probs.sum()
dist_summary = s8u.plot_categorical_distribution(toy_support, toy_probs, title='One categorical return distribution on a fixed support')
print('Expected return from the categorical distribution:', round(dist_summary['expected_value'], 3))


### Task 4

**Implement the categorical projection and a distributional dueling head (25 min)**

This task has two parts.

First, implement the C51 projection. After the Bellman update shifts and rescales the target distribution, its probability mass must be projected back onto the fixed support used by the network. That projection is what keeps the output representation stable from one update to the next.

Second, build a network head that predicts logits over atoms for every action. The dueling idea still applies here as well, except that the value and advantage streams now predict **atom logits** rather than single scalar values.


In [ ]:
def project_categorical_distribution(
    support: torch.Tensor,
    next_probs: torch.Tensor,
    rewards: torch.Tensor,
    dones: torch.Tensor,
    gamma: float,
    n_step: int = 1,
) -> torch.Tensor:
########################################################################
    batch_size = next_probs.shape[0]
    num_atoms = support.numel()
    v_min = float(support[0].item())
    v_max = float(support[-1].item())
    delta_z = float((v_max - v_min) / (num_atoms - 1))

    tz = rewards.unsqueeze(1) + (gamma**n_step) * (1.0 - dones.unsqueeze(1)) * support.unsqueeze(0)
    tz = tz.clamp(v_min, v_max)
    b = (tz - v_min) / delta_z
    lower = b.floor().long()
    upper = b.ceil().long()

    projected = torch.zeros_like(next_probs)
    for batch_idx in range(batch_size):
        for atom_idx in range(num_atoms):
            l = int(lower[batch_idx, atom_idx].item())
            u = int(upper[batch_idx, atom_idx].item())
            p = float(next_probs[batch_idx, atom_idx].item())
            if l == u:
                projected[batch_idx, l] += p
            else:
                projected[batch_idx, l] += p * (u - b[batch_idx, atom_idx])
                projected[batch_idx, u] += p * (b[batch_idx, atom_idx] - l)
    return projected
########################################################################


class CategoricalDuelingNetwork(nn.Module):
    def __init__(
        self,
        obs_shape: tuple[int, int, int],
        num_actions: int,
        num_atoms: int = 51,
        v_min: float = -10.0,
        v_max: float = 10.0,
        linear_layer_cls=nn.Linear,
    ):
        super().__init__()
        c, h, w = obs_shape
        self.num_actions = int(num_actions)
        self.num_atoms = int(num_atoms)
        self.register_buffer('support', torch.linspace(v_min, v_max, num_atoms))

        self.features = nn.Sequential(
            nn.Conv2d(c, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, c, h, w)
            feature_dim = int(np.prod(self.features(dummy).shape[1:]))

        self.value_stream = nn.Sequential(
            linear_layer_cls(feature_dim, 512),
            nn.ReLU(),
            linear_layer_cls(512, num_atoms),
        )
        self.advantage_stream = nn.Sequential(
            linear_layer_cls(feature_dim, 512),
            nn.ReLU(),
            linear_layer_cls(512, num_actions * num_atoms),
        )

    def forward_logits(self, obs: torch.Tensor) -> torch.Tensor:
        features = self.features(obs).flatten(start_dim=1)
        value_logits = self.value_stream(features).view(-1, 1, self.num_atoms)
        advantage_logits = self.advantage_stream(features).view(-1, self.num_actions, self.num_atoms)
        return value_logits + advantage_logits - advantage_logits.mean(dim=1, keepdim=True)

    def forward_probs(self, obs: torch.Tensor) -> torch.Tensor:
        return torch.softmax(self.forward_logits(obs), dim=-1)

    def q_values(self, obs: torch.Tensor) -> torch.Tensor:
        probs = self.forward_probs(obs)
        return torch.sum(probs * self.support.view(1, 1, -1), dim=-1)


In [ ]:
support = torch.linspace(-10.0, 10.0, 11, device=DEVICE)
next_probs = torch.softmax(torch.tensor([[0.2, 0.5, 1.0, 1.5, 2.0, 2.4, 2.0, 1.6, 1.1, 0.6, 0.3]], device=DEVICE), dim=1)
projected = project_categorical_distribution(
    support=support,
    next_probs=next_probs,
    rewards=torch.tensor([1.5], device=DEVICE),
    dones=torch.tensor([0.0], device=DEVICE),
    gamma=0.99,
    n_step=3,
)

s8u.plot_distribution_projection(
    old_support=(1.5 + (0.99**3) * support).detach().cpu().numpy(),
    old_probs=next_probs[0].detach().cpu().numpy(),
    new_support=support.detach().cpu().numpy(),
    new_probs=projected[0].detach().cpu().numpy(),
)

categorical_net = CategoricalDuelingNetwork(PONG_OBS_SHAPE, PONG_NUM_ACTIONS, num_atoms=51).to(DEVICE)
dummy_obs = torch.rand(2, *PONG_OBS_SHAPE, device=DEVICE)
probs = categorical_net.forward_probs(dummy_obs)
q_values = categorical_net.q_values(dummy_obs)
print('Categorical probabilities shape:', tuple(probs.shape))
print('Expected Q-values shape:', tuple(q_values.shape))


## Putting the Rainbow pieces together


At this point the main Rainbow ingredients are all on the table. The remaining step is to see how they interact in one update:

- prioritized replay provides `weights` and sampling indices,
- Double DQN chooses the next action with the online network,
- the target network evaluates that chosen action,
- the target is multi-step,
- the network uses noisy layers and a dueling distributional head,
- and the loss is computed against a projected categorical target distribution.

The code below is still only a draft integration block, but it shows the structure of a Rainbow-style training step much more clearly once the components above are already familiar.

Reference paper: [Hessel et al. (2018), *Rainbow: Combining Improvements in Deep Reinforcement Learning*](https://arxiv.org/abs/1710.02298).


In [ ]:
class RainbowNetwork(CategoricalDuelingNetwork):
    def __init__(self, obs_shape: tuple[int, int, int], num_actions: int, num_atoms: int = 51, v_min: float = -10.0, v_max: float = 10.0):
        super().__init__(
            obs_shape=obs_shape,
            num_actions=num_actions,
            num_atoms=num_atoms,
            v_min=v_min,
            v_max=v_max,
            linear_layer_cls=NoisyLinear,
        )

    def reset_noise(self):
        for module in self.modules():
            if isinstance(module, NoisyLinear):
                module.reset_noise()


def rainbow_train_step(
    online_net: RainbowNetwork,
    target_net: RainbowNetwork,
    optimizer,
    batch: dict[str, torch.Tensor],
    gamma: float,
    n_step: int = 3,
    max_grad_norm: float = 10.0,
) -> dict[str, float | np.ndarray]:
########################################################################
    online_net.train()
    target_net.train()
    online_net.reset_noise()
    target_net.reset_noise()

    with torch.no_grad():
        next_q = online_net.q_values(batch['next_obs'])
        next_actions = torch.argmax(next_q, dim=1)
        next_probs_all = target_net.forward_probs(batch['next_obs'])
        next_probs = next_probs_all[torch.arange(next_probs_all.shape[0], device=batch['next_obs'].device), next_actions]
        target_dist = project_categorical_distribution(
            support=online_net.support,
            next_probs=next_probs,
            rewards=batch['rewards'],
            dones=batch['dones'],
            gamma=gamma,
            n_step=n_step,
        )

    logits = online_net.forward_logits(batch['obs'])
    chosen_logits = logits[torch.arange(logits.shape[0], device=batch['obs'].device), batch['actions']]
    log_probs = torch.log_softmax(chosen_logits, dim=1)
    per_sample_loss = -(target_dist * log_probs).sum(dim=1)

    weights = batch.get('weights', torch.ones_like(per_sample_loss))
    loss = (weights * per_sample_loss).mean()

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(online_net.parameters(), max_grad_norm)
    optimizer.step()

    priorities = per_sample_loss.detach().cpu().numpy() + 1e-6
    return {
        'loss': float(loss.item()),
        'mean_q': float(online_net.q_values(batch['obs']).mean().item()),
        'priorities': priorities,
    }
########################################################################


In [ ]:
rainbow_net = RainbowNetwork(PONG_OBS_SHAPE, PONG_NUM_ACTIONS).to(DEVICE)
rainbow_target = RainbowNetwork(PONG_OBS_SHAPE, PONG_NUM_ACTIONS).to(DEVICE)
rainbow_target.load_state_dict(rainbow_net.state_dict())
rainbow_optimizer = optim.Adam(rainbow_net.parameters(), lr=1e-4)

fake_batch = {
    'obs': torch.rand(4, *PONG_OBS_SHAPE, device=DEVICE),
    'actions': torch.tensor([0, 2, 3, 5], dtype=torch.long, device=DEVICE),
    'rewards': torch.tensor([1.0, -1.0, 0.5, 0.0], dtype=torch.float32, device=DEVICE),
    'next_obs': torch.rand(4, *PONG_OBS_SHAPE, device=DEVICE),
    'dones': torch.tensor([0.0, 0.0, 1.0, 0.0], dtype=torch.float32, device=DEVICE),
    'weights': torch.tensor([1.0, 0.8, 1.0, 0.6], dtype=torch.float32, device=DEVICE),
}

rainbow_stats = rainbow_train_step(rainbow_net, rainbow_target, rainbow_optimizer, fake_batch, gamma=0.99, n_step=3)
print('Rainbow training-step stats:', {k: v for k, v in rainbow_stats.items() if k != 'priorities'})
print('Suggested priority updates:', np.round(rainbow_stats['priorities'], 4))


## Bridge to Full Rainbow Experiments

This notebook stops at the component level on purpose. The next step is not to invent new machinery, but to plug these pieces back into the Pong training loop from Practice 7:

- replace the uniform replay buffer with prioritized replay,
- replace the DQN head with the Rainbow network,
- replace epsilon-greedy exploration with noisy layers,
- use multi-step returns when storing replay entries,
- and optimize the categorical loss with the Double DQN target selection rule.

Once those pieces are connected, the resulting implementation is much closer to a real Rainbow agent. The important thing is that by this point every line in that combined update already has a local explanation.
